In [1]:
import re
import time
from pathlib import Path
import numpy as np
import pandas as pd
 
EXTRACT_DIR = "/kaggle/input/datasets/liamguske/allscores"
OUTPUT_PATH = "/kaggle/working/submission.csv"
 
t0 = time.time()
 
# Load all submissions, parse Kaggle public score from filename
files_and_scores = []
for f in sorted(Path(EXTRACT_DIR).glob("*.csv")):
    m = re.match(r"^(\d+\.\d+)", f.name)
    if m:
        files_and_scores.append((f, float(m.group(1))))
 
print(f"Loaded {len(files_and_scores)} submissions")
 
# Canonical TrackID ordering from the first file
first_df  = pd.read_csv(files_and_scores[0][0]).set_index("TrackID")
trackids  = first_df.index
 
# Build S (N × K) matrix
s_cols = []
P = []
for f, score in files_and_scores:
    df = pd.read_csv(f).set_index("TrackID").reindex(trackids)
    s_cols.append(2 * df["Predictor"].values.astype(np.float64) - 1)
    P.append(score)
 
S = np.column_stack(s_cols)
P = np.array(P, dtype=np.float64)
N, K = S.shape
print(f"S shape: {N:,} × {K}")
print(f"Kaggle scores: min={P.min():.3f}  max={P.max():.3f}  mean={P.mean():.3f}")
 
# S^T x via the Kaggle-score identity
STx = N * (2 * P - 1)
 
# S^T S and least-squares solution
STS  = S.T @ S                          
a_LS, residuals, rank, sv = np.linalg.lstsq(STS, STx, rcond=None)
print(f"\nRank(S^T S) = {rank} / {K}")
print(f"Condition number = {sv[0] / sv[-1]:.2e}")
 
# Print weights for inspection
print(f"\n{'Submission':<20} {'Kaggle P':>10}  {'Weight':>10}")
print("─" * 44)
for (f, score), w in sorted(zip(files_and_scores, a_LS),
                             key=lambda x: x[1], reverse=True):
    print(f"  {f.name:<18} {score:>10.3f}  {w:>+10.4f}")
 
# Apply weights -> continuous ensemble scores -> top-3 per user
s_ensemble = S @ a_LS                 
 
out = pd.DataFrame({"TrackID": trackids, "score": s_ensemble})
out[["userID", "trackID"]] = out["TrackID"].str.split("_", n=1, expand=True)
out["rank"]      = out.groupby("userID")["score"].rank(method="first",
                                                       ascending=False)
out["Predictor"] = (out["rank"] <= 3).astype(int)
 
final = out[["TrackID", "Predictor"]].sort_values("TrackID")
final.to_csv(OUTPUT_PATH, index=False)
 
likes = int(final["Predictor"].sum())
print(f"\nSaved: {OUTPUT_PATH}")
print(f"Rows: {len(final):,}  Likes: {likes:,} ({likes/len(final)*100:.1f}%)")
print(f"Runtime: {time.time() - t0:.1f}s")

Loaded 45 submissions
S shape: 120,000 × 45
Kaggle scores: min=0.311  max=0.889  mean=0.746

Rank(S^T S) = 45 / 45
Condition number = 5.13e+03

Submission             Kaggle P      Weight
────────────────────────────────────────────
  0.889.csv               0.889     +0.2223
  0.882.csv               0.882     +0.1518
  0.856.csv               0.856     +0.1259
  0.826.csv               0.826     +0.0842
  0.860.csv               0.860     +0.0806
  0.843.csv               0.843     +0.0689
  0.627.csv               0.627     +0.0653
  0.885.csv               0.885     +0.0652
  0.821.csv               0.821     +0.0626
  0.808.csv               0.808     +0.0535
  0.854.csv               0.854     +0.0515
  0.699.csv               0.699     +0.0484
  0.626.csv               0.626     +0.0469
  0.795.csv               0.795     +0.0383
  0.724 (2).csv           0.724     +0.0321
  0.822.csv               0.822     +0.0297
  0.619.csv               0.619     +0.0279
  0.852.csv        